# Lezione 01 - Introduzione agli Agenti AI

Benvenuto alla prima lezione del corso **Agenti AI per Principianti**!

Un **agente AI** è un programma che utilizza un modello di linguaggio di grandi dimensioni (LLM) come motore di ragionamento e può compiere *azioni* nel mondo reale — chiamare API, interrogare database o eseguire codice — per raggiungere un obiettivo per conto di un utente.

In questo notebook costruirai il tuo primo agente: un **Agente di Viaggi** che consiglia destinazioni per le vacanze. Durante il percorso imparerai a:

1. Connetterti al servizio Azure AI Foundry Agent utilizzando il **Microsoft Agent Framework**.
2. Fornire all'agente un **strumento** — una semplice funzione Python che può chiamare.
3. Eseguire l'agente e ispezionarne la risposta.
4. Trasmettere la risposta dell'agente token per token.


## Configurazione

Prima di eseguire questo notebook, assicurati di avere:

1. **Un progetto Azure AI Foundry** con un modello chat distribuito (ad esempio `gpt-4o-mini`).
2. **Effettuato l'accesso con l'Azure CLI** — esegui `az login` nel tuo terminale.
3. **Impostato le variabili d'ambiente richieste:**
   - `AZURE_AI_PROJECT_ENDPOINT` — il tuo endpoint del progetto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — il nome del tuo modello distribuito.

La cella qui sotto installa i pacchetti Python necessari.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Creare il tuo primo agente

Un agente ha bisogno di due cose:

- **Istruzioni** che gli dicano *chi è* e *come comportarsi* (un prompt di sistema).
- **Strumenti** — funzioni Python decorate con `@tool` che l'agente può chiamare per recuperare informazioni o eseguire azioni.

Di seguito definiamo uno strumento semplice che restituisce una lista di destinazioni di vacanza popolari. L'agente userà questo strumento quando un utente chiede consigli di viaggio.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Risposte in Streaming

Per un'esperienza più interattiva puoi **trasmettere in streaming** la risposta dell'agente. Invece di aspettare la risposta completa, l'agente fornisce porzioni di testo man mano che vengono generate. Questo è particolarmente utile nelle interfacce chat dove vuoi mostrare l'output in tempo reale.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Riepilogo

In questa lezione hai imparato a:

- **Creare un provider** che si connette ad Azure AI Foundry Agent Service tramite `AzureAIProjectAgentProvider`.
- **Definire uno strumento** utilizzando il decoratore `@tool` in modo che l'agente possa chiamare le tue funzioni Python.
- **Eseguire l'agente** con un messaggio utente e stampare la sua risposta.
- **Trasmettere le risposte** per output in tempo reale.

Nella prossima lezione esploreremo i framework agentici in modo più approfondito e impareremo come fornire agli agenti strumenti più potenti e capacità di ragionamento a più passaggi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avvertenza**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per l’accuratezza, si prega di considerare che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella sua lingua madre deve essere considerato la fonte autorevole. Per informazioni di carattere critico si raccomanda una traduzione professionale effettuata da traduttori umani. Non ci assumiamo responsabilità per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
